In [0]:

# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from datetime import datetime, timezone
from uuid import uuid4

from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType


"""
Ingest monthly semiconductor production-lot files into the Bronze layer.

The pipeline processes files currently available in the external landing
volume and retains source-file metadata for reconciliation, lineage, and
failure investigation.
"""

# Managed Bronze table containing raw production-lot records.
TARGET_TABLE = "semiconplus_portfolio.bronze.production_lots"

# External ADLS landing directory containing monthly production-lot CSV files.
SOURCE_PATH = (
    "/Volumes/semiconplus_portfolio/"
    "landing/external_source/production_lots"
)

# Stores Auto Loader schema-tracking information.
SCHEMA_LOCATION = (
    "/Volumes/semiconplus_portfolio/"
    "monitoring/checkpoints/autoloader/production_lots/schema"
)

# Stores processed-file state for incremental and idempotent ingestion.
CHECKPOINT_LOCATION = (
    "/Volumes/semiconplus_portfolio/"
    "monitoring/checkpoints/autoloader/production_lots/checkpoint"
)

# Identifies records written by the current pipeline execution.
PIPELINE_RUN_ID = str(uuid4())

# Used to record the pipeline execution duration.
PIPELINE_START_TIME = datetime.now(timezone.utc)

print(f"Pipeline run ID: {PIPELINE_RUN_ID}")
print(f"Pipeline start UTC: {PIPELINE_START_TIME.isoformat()}")
print(f"Source path: {SOURCE_PATH}")
print(f"Target table: {TARGET_TABLE}")

In [0]:


# BLOCK 2 — SOURCE SCHEMA
# ===================================================

"""
Define the expected production-lot source structure.

Source values remain strings in Bronze to preserve the original data.
Business data types and validation rules are applied in Silver, where
invalid records can be isolated without losing the source payload.
"""

production_lot_schema = StructType(
    [
        StructField("lot_id", StringType(), True),
        StructField("production_date", StringType(), True),
        StructField("device_id", StringType(), True),
        StructField("product_group_id", StringType(), True),
        StructField("site_id", StringType(), True),
        StructField("equipment_id", StringType(), True),
        StructField("start_timestamp_utc", StringType(), True),
        StructField("quantity_started", StringType(), True),
        StructField("quantity_passed", StringType(), True),
        StructField("quantity_failed", StringType(), True),
        StructField("actual_yield", StringType(), True),
        StructField("test_program_revision", StringType(), True),
        StructField("source_system", StringType(), True),
    ]
)

print(production_lot_schema.simpleString())


In [0]:
# ===================================================
# BLOCK 3 — SOURCE PREFLIGHT VALIDATION
# ===================================================

"""
Confirm that the expected monthly source files are accessible before
starting ingestion.

The pipeline stops when the source-file count or total file size does
not match the approved five-year dataset baseline.
"""

# Include only CSV source files in the ingestion baseline.
source_files = [
    file_info
    for file_info in dbutils.fs.ls(SOURCE_PATH)
    if file_info.name.lower().endswith(".csv")
]

source_file_count = len(source_files)
source_total_bytes = sum(file_info.size for file_info in source_files)

print(f"CSV files discovered: {source_file_count}")
print(f"Total source bytes: {source_total_bytes:,}")

assert source_file_count == 60, (
    f"Expected 60 monthly CSV files, but found {source_file_count}."
)
assert source_total_bytes > 0, "Source CSV files are empty."

display(
    spark.createDataFrame(
        [
            (file_info.name, file_info.path, file_info.size)
            for file_info in source_files
        ],
        ["file_name", "file_path", "size_bytes"],
    ).orderBy("file_name")
)


In [0]:
# ===================================================
# BLOCK 4 — AUTO LOADER SOURCE
# ===================================================

"""
Configure incremental ingestion from the external production-lot landing
directory.

Unexpected source fields are retained in _rescued_data for investigation
instead of being silently discarded.
"""

bronze_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", SCHEMA_LOCATION)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .schema(production_lot_schema)
    .load(SOURCE_PATH)
)

print("Auto Loader source configured.")
print(f"Streaming input: {bronze_stream.isStreaming}")

assert bronze_stream.isStreaming, "Auto Loader source is not streaming."

In [0]:
# ===================================================
# BLOCK 5 — OPERATIONAL METADATA
# ===================================================

"""
Attach ingestion metadata used for source reconciliation, operational
monitoring, lineage, and incident investigation.
"""

bronze_with_metadata = bronze_stream.select(
    "*",

    # Capture the physical source-file details supplied by Databricks.
    F.col("_metadata.file_path").alias("_source_file_path"),
    F.col("_metadata.file_name").alias("_source_file_name"),
    F.col("_metadata.file_modification_time").alias(
        "_source_file_modification_time"
    ),

    # Record when the row entered the Bronze table.
    F.current_timestamp().alias("_ingested_at_utc"),

    # Associate the row with the current pipeline execution.
    F.lit(PIPELINE_RUN_ID).alias("_pipeline_run_id"),
)

print("Operational metadata columns attached.")


In [0]:
# ===================================================
# BLOCK 6 — BRONZE DELTA WRITE
# ===================================================

"""
Append newly discovered source files to the managed Bronze Delta table.

The available-now trigger clears the current file backlog and terminates
the stream. The checkpoint prevents previously processed files from being
appended again during subsequent executions.
"""

ingestion_query = (
    bronze_with_metadata.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_LOCATION)
    .option("mergeSchema", "false")
    .trigger(availableNow=True)
    .toTable(TARGET_TABLE)
)

# Prevent downstream validation from starting before ingestion completes.
ingestion_query.awaitTermination()

PIPELINE_END_TIME = datetime.now(timezone.utc)
PIPELINE_DURATION_SECONDS = (
    PIPELINE_END_TIME - PIPELINE_START_TIME
).total_seconds()

print("Bronze ingestion completed.")
print(f"Pipeline end UTC: {PIPELINE_END_TIME.isoformat()}")
print(f"Duration: {PIPELINE_DURATION_SECONDS:.2f} seconds")

In [0]:
# ===================================================
# BLOCK 7 — POST-INGESTION CONTROLS
# ===================================================

"""
Run the minimum control checks required before detailed data validation.

The pipeline fails when the Bronze record count or required lineage
columns do not match the approved ingestion contract.
"""

bronze_df = spark.table(TARGET_TABLE)

bronze_row_count = bronze_df.count()
bronze_column_count = len(bronze_df.columns)

print(f"Bronze row count: {bronze_row_count:,}")
print(f"Bronze column count: {bronze_column_count}")

assert bronze_row_count == 18_126, (
    f"Expected 18,126 Bronze records, but found {bronze_row_count:,}."
)

required_metadata_columns = {
    "_source_file_path",
    "_source_file_name",
    "_source_file_modification_time",
    "_ingested_at_utc",
    "_pipeline_run_id",
    "_rescued_data",
}

missing_metadata_columns = required_metadata_columns.difference(
    bronze_df.columns
)

assert not missing_metadata_columns, (
    f"Missing metadata columns: {sorted(missing_metadata_columns)}"
)

print("Immediate Bronze controls passed.")